# Exploration and Simple Model

This notebook uses the cleaned wind turbine dataset for basic analysis, visualizations, and simple power prediction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path('../datasets')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / 'wind_turbine_train_clean.csv')
display(df.head())

In [ ]:
corr_cols = ['Power','Temp_2m','RelHum_2m','DP_2m','WS_10m','WS_100m','WD_10m','WD_100m','WG_10m','wind_speed_difference_100m_10m','wind_speed_100m_cubed']
corr = df[corr_cols].corr()
corr.to_csv(DATA_DIR / 'wind_turbine_correlation_matrix.csv')
display(corr)

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df['WS_100m'], df['Power'], s=2, alpha=0.2)
plt.xlabel('Wind speed at 100m')
plt.ylabel('Power')
plt.title('Wind speed at 100m vs power')
plt.tight_layout()
plt.savefig(FIG_DIR / 'wind_speed_100m_vs_power.png', dpi=150)
plt.show()

In [ ]:
monthly = pd.read_csv(DATA_DIR / 'wind_turbine_monthly_summary.csv')
plt.figure(figsize=(8,5))
monthly.groupby('month')['avg_power'].mean().plot(kind='bar')
plt.xlabel('Month')
plt.ylabel('Average power')
plt.title('Average power by month')
plt.tight_layout()
plt.savefig(FIG_DIR / 'average_power_by_month.png', dpi=150)
plt.show()

In [ ]:
features = ['Temp_2m','RelHum_2m','DP_2m','WS_10m','WS_100m','WG_10m',
            'wind_speed_difference_100m_10m','wind_speed_100m_squared','wind_speed_100m_cubed',
            'wind_direction_100m_sin','wind_direction_100m_cos','hour','month','Location']

model_df = df.dropna(subset=features + ['Power']).copy()
X = model_df[features]
y = model_df['Power']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=12)
}

results = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    trained_models[name] = model
    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_val, pred),
        'RMSE': mean_squared_error(y_val, pred) ** 0.5,
        'R2': r2_score(y_val, pred)
    })

metrics = pd.DataFrame(results)
metrics.to_csv(DATA_DIR / 'model_metrics.csv', index=False)
display(metrics)

In [ ]:
rf = trained_models['RandomForestRegressor']
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.to_csv(DATA_DIR / 'random_forest_feature_importance.csv', index=False)
display(feature_importance)

plt.figure(figsize=(8,5))
feature_importance.head(10).plot(kind='barh', x='feature', y='importance', legend=False)
plt.xlabel('Importance')
plt.title('Random forest feature importance')
plt.tight_layout()
plt.savefig(FIG_DIR / 'random_forest_feature_importance.png', dpi=150)
plt.show()

In [ ]:
test = pd.read_csv(DATA_DIR / 'wind_turbine_test_clean.csv')
test['predicted_power'] = rf.predict(test[features])
test.to_csv(DATA_DIR / 'wind_turbine_test_with_predictions.csv', index=False)
display(test[['Time','Location','WS_100m','predicted_power']].head())